In [3]:
!pip install rasterio pyproj tqdm -q
from google.colab import drive; drive.mount('/content/drive')
"""
snoco_downloader.py
--------------------
Downloads Snohomish County aerial imagery (≤9 inch resolution) from the
ArcGIS ImageServer for a user-defined AOI (default: Edmonds city boundary).

Designed for Google Colab CPU runtime (12 GB RAM).
- Tiles each year into 4000×4000 px chunks
- Merges tiles into a single GeoTIFF per year/band-set
- Writes final files to Google Drive
- 2021 gets a dual pass: RGB + NIR (band 3) as separate outputs
- 2018 gets RGB only (band 4 is a mask/nodata layer, not NIR)
- 2012 uses EPSG:2926 (WA State Plane North, different realization)

Usage in Colab:
    !pip install requests rasterio pyproj tqdm
    # Mount Drive first, then run.
"""

import os
import math
import time
import requests
import rasterio
from rasterio.crs import CRS
from rasterio.merge import merge
from pyproj import Transformer
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# ─────────────────────────────────────────────────────────────────────────────
# USER CONFIG
# ─────────────────────────────────────────────────────────────────────────────

# Edmonds AOI in WGS84 — adjust if you want a tighter clip
AOI_WGS84 = {
    "xmin": -122.404,
    "ymin":   47.783,
    "xmax": -122.316,
    "ymax":   47.828,
}

# Output root on Google Drive (change to your mount path)
OUTPUT_ROOT = "/content/drive/MyDrive/treedata/Full_Image/SnoCo/v2"

# Tile size in pixels (safe under server 4100 limit; leaves headroom)
TILE_PX = 512

# Seconds to wait between tile requests (be a polite client)
REQUEST_DELAY = 0.5

# Max retries per tile
MAX_RETRIES = 3

WORKER_STAGGER = 0.2

N_WORKERS = 4

# ─────────────────────────────────────────────────────────────────────────────
# LAYER REGISTRY
# All pixel sizes in feet (native server units).
# band_ids: list of band indices to request; None = all bands (server default)
# nir_band_id: if set, a second NIR-only pass is run
# src_epsg: native CRS of this service
# ─────────────────────────────────────────────────────────────────────────────

LAYERS = [
    {
        "year":        2012,
        "label":       "SnoCo 2012",
        "url":         "https://gis.snoco.org/img/rest/services/Imagery/Aerial_2012/ImageServer",
        "pixel_ft":    0.75,   # 9 inch
        "bands":       3,
        "band_ids":    [0, 1, 2],
        "nir_band_id": None,
        "src_epsg":    2926,   # NOTE: different from all other years
        "out_rgb":     "snoco_2012_rgb.tif",
        "out_nir":     None,
    },
    {
        "year":        2016,
        "label":       "SnoCo 2016",
        "url":         "https://gis.snoco.org/img/rest/services/Imagery/Aerial_2016/ImageServer",
        "pixel_ft":    0.5,    # 6 inch
        "bands":       4,
        "band_ids":    [0, 1, 2],
        "nir_band_id": 3,
        "src_epsg":    2285,
        "out_rgb":     "snoco_2016_rgb.tif",
        "out_nir":     "snoco_2016_nir.tif",
    },


]


# ─────────────────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def wgs84_to_projected(xmin, ymin, xmax, ymax, epsg):
    """Reproject WGS84 bounding box to a projected CRS."""
    transformer = Transformer.from_crs("EPSG:4326", f"EPSG:{epsg}", always_xy=True)
    x0, y0 = transformer.transform(xmin, ymin)
    x1, y1 = transformer.transform(xmax, ymax)
    return x0, y0, x1, y1


def build_tile_grid(xmin, ymin, xmax, ymax, pixel_ft, tile_px):
    """
    Returns a list of (tile_xmin, tile_ymin, tile_xmax, tile_ymax, size_x, size_y)
    tuples covering the AOI without gaps.
    size_x/size_y are the pixel dimensions of this tile (last tiles may be smaller).
    """
    tile_ft = tile_px * pixel_ft          # ground extent covered per tile
    cols = math.ceil((xmax - xmin) / tile_ft)
    rows = math.ceil((ymax - ymin) / tile_ft)

    tiles = []
    for row in range(rows):
        for col in range(cols):
            tx0 = xmin + col * tile_ft
            ty0 = ymin + row * tile_ft
            tx1 = min(tx0 + tile_ft, xmax)
            ty1 = min(ty0 + tile_ft, ymax)
            # Pixel dimensions for this tile
            sx = round((tx1 - tx0) / pixel_ft)
            sy = round((ty1 - ty0) / pixel_ft)
            if sx > 0 and sy > 0:
                tiles.append((tx0, ty0, tx1, ty1, sx, sy))

    return tiles, cols, rows


def fetch_tile(url, bbox, size_x, size_y, band_ids, epsg, retries=MAX_RETRIES):
    """
    Calls exportImage on the ArcGIS ImageServer and returns raw TIFF bytes.
    bbox = (xmin, ymin, xmax, ymax) in the service's native CRS.
    """
    params = {
        "bbox":                 f"{bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]}",
        "bboxSR":               epsg,
        "size":                 f"{size_x},{size_y}",
        "imageSR":              epsg,
        "format":               "tiff",
        "pixelType":            "U8",
        "noData":               "",
        "noDataInterpretation": "esriNoDataMatchAny",
        "interpolation":        "RSP_BilinearInterpolation",
        "bandIds":              ",".join(str(b) for b in band_ids),
        "f":                    "image",
    }

    for attempt in range(retries):
        try:
            r = requests.get(f"{url}/exportImage", params=params, timeout=120)
            r.raise_for_status()
            # ArcGIS returns errors as JSON even when format=image
            if r.headers.get("Content-Type", "").startswith("application/json"):
                raise ValueError(f"Server returned JSON error: {r.text[:300]}")
            return r.content
        except Exception as e:
            if attempt < retries - 1:
                wait = 2 ** attempt
                print(f"      Retry {attempt+1}/{retries-1} after {wait}s — {e}")
                time.sleep(wait)
            else:
                raise


def save_tile_bytes(tile_bytes, tmp_path):
    """Write raw TIFF bytes to a temp file and return the path."""
    with open(tmp_path, "wb") as f:
        f.write(tile_bytes)
    return tmp_path


def merge_tiles_to_geotiff(tile_paths, out_path, epsg, n_bands):
    datasets = [rasterio.open(p) for p in tile_paths]
    mosaic, transform = merge(datasets)

    crs = CRS.from_epsg(epsg)
    meta = datasets[0].meta.copy()
    meta.update({
        "driver":     "GTiff",
        "height":     mosaic.shape[1],
        "width":      mosaic.shape[2],
        "count":      n_bands,
        "transform":  transform,
        "crs":        crs,
        "compress":   "lzw",
        "tiled":      True,
        "blockxsize": 512,
        "blockysize": 512,
        "BIGTIFF":    "YES",
    })

    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with rasterio.open(out_path, "w", **meta) as dst:
        dst.write(mosaic)

    for ds in datasets:
        ds.close()


def _fetch_one_tile(args):
    """
    Worker function — called by ThreadPoolExecutor.
    Fetches a single tile and writes it to tmp_path.
    Returns tmp_path on success, raises on unrecoverable failure.

    args = (i, tx0, ty0, tx1, ty1, sx, sy, url, band_ids, epsg, tmp_path, worker_id)
    """
    i, tx0, ty0, tx1, ty1, sx, sy, url, band_ids, epsg, tmp_path, worker_id = args

    # Resume: skip tiles already written in a previous interrupted run
    if os.path.exists(tmp_path) and os.path.getsize(tmp_path) > 0:
        return tmp_path

    # Stagger workers slightly so they don't all fire at t=0
    time.sleep(worker_id * WORKER_STAGGER)

    tile_bytes = fetch_tile(
        url      = url,
        bbox     = (tx0, ty0, tx1, ty1),
        size_x   = sx,
        size_y   = sy,
        band_ids = band_ids,
        epsg     = epsg,
    )
    save_tile_bytes(tile_bytes, tmp_path)
    return tmp_path


def download_layer_pass(layer, aoi_proj, band_ids, out_path, pass_label):
    """
    Full tile-and-merge pass for one layer + one set of band_ids.
    Tiles are fetched in parallel using N_WORKERS threads.
    aoi_proj = (xmin, ymin, xmax, ymax) in the layer's native CRS.
    """
    xmin, ymin, xmax, ymax = aoi_proj
    pixel_ft = layer["pixel_ft"]
    epsg     = layer["src_epsg"]
    n_bands  = len(band_ids)

    tiles, cols, rows = build_tile_grid(xmin, ymin, xmax, ymax, pixel_ft, TILE_PX)
    total = len(tiles)
    print(f"\n  {pass_label}: {total} tiles ({cols} cols × {rows} rows), "
          f"pixel={pixel_ft} ft, bands={band_ids}, workers={N_WORKERS}")

    tmp_dir = f"/tmp/snoco_{layer['year']}_{pass_label.replace(' ','_')}"
    os.makedirs(tmp_dir, exist_ok=True)

    # Build work items — each tile gets a deterministic path assigned up front
    # so workers never race to claim the same file
    work_items = []
    for i, (tx0, ty0, tx1, ty1, sx, sy) in enumerate(tiles):
        tmp_path  = os.path.join(tmp_dir, f"tile_{i:04d}.tif")
        worker_id = i % N_WORKERS   # used for stagger offset only
        work_items.append(
            (i, tx0, ty0, tx1, ty1, sx, sy,
             layer["url"], band_ids, epsg, tmp_path, worker_id)
        )

    # Ordered result list — filled by index as futures complete
    tile_paths = [None] * total
    failed     = []

    with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
        future_to_idx = {pool.submit(_fetch_one_tile, item): item[0]
                         for item in work_items}

        with tqdm(total=total, desc=f"    {pass_label}", unit="tile") as pbar:
            for future in as_completed(future_to_idx):
                idx = future_to_idx[future]
                try:
                    tile_paths[idx] = future.result()
                except Exception as exc:
                    print(f"\n  ERROR tile {idx:04d}: {exc}")
                    failed.append(idx)
                finally:
                    pbar.update(1)

    if failed:
        raise RuntimeError(
            f"{len(failed)} tile(s) failed for {pass_label}: indices {failed}\n"
            "Fix the issue and re-run — successful tiles are cached on disk."
        )

    print(f"  Merging {total} tiles → {out_path}")
    merge_tiles_to_geotiff(tile_paths, out_path, epsg, n_bands)

    # Clean up temp tiles
    for p in tile_paths:
        if p and os.path.exists(p):
            os.remove(p)
    try:
        os.rmdir(tmp_dir)
    except OSError:
        pass  # non-empty dir on partial run — leave it for next resume
    print(f"  ✓ Done: {out_path}")


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────

def main():
    os.makedirs(OUTPUT_ROOT, exist_ok=True)

    for layer in LAYERS:
        print(f"\n{'='*60}")
        print(f"  {layer['label']}  ({layer['year']})")
        print(f"  {layer['url']}")
        print(f"{'='*60}")

        epsg = layer["src_epsg"]

        # Reproject AOI into this layer's native CRS
        aoi_proj = wgs84_to_projected(
            AOI_WGS84["xmin"], AOI_WGS84["ymin"],
            AOI_WGS84["xmax"], AOI_WGS84["ymax"],
            epsg
        )
        print(f"  AOI (EPSG:{epsg}): "
              f"{aoi_proj[0]:.0f},{aoi_proj[1]:.0f}  →  "
              f"{aoi_proj[2]:.0f},{aoi_proj[3]:.0f}")

        # ── RGB pass ──────────────────────────────────────────────────────
        out_rgb = os.path.join(OUTPUT_ROOT, layer["out_rgb"])
        if os.path.exists(out_rgb):
            print(f"  RGB already exists, skipping: {out_rgb}")
        else:
            download_layer_pass(
                layer      = layer,
                aoi_proj   = aoi_proj,
                band_ids   = layer["band_ids"],
                out_path   = out_rgb,
                pass_label = "RGB",
            )

        # ── NIR pass (optional) ───────────────────────────────────────────
        if layer["nir_band_id"] is not None:
            out_nir = os.path.join(OUTPUT_ROOT, layer["out_nir"])
            if os.path.exists(out_nir):
                print(f"  NIR already exists, skipping: {out_nir}")
            else:
                download_layer_pass(
                    layer      = layer,
                    aoi_proj   = aoi_proj,
                    band_ids   = [layer["nir_band_id"]],
                    out_path   = out_nir,
                    pass_label = "NIR",
                )

    print(f"\n{'='*60}")
    print("  All downloads complete.")
    print(f"{'='*60}\n")


if __name__ == "__main__":
    main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

  SnoCo 2012  (2012)
  https://gis.snoco.org/img/rest/services/Imagery/Aerial_2012/ImageServer
  AOI (EPSG:2926): 1254301,289552  →  1276248,305535
  RGB already exists, skipping: /content/drive/MyDrive/treedata/Full_Image/SnoCo/v2/snoco_2012_rgb.tif

  SnoCo 2016  (2016)
  https://gis.snoco.org/img/rest/services/Imagery/Aerial_2016/ImageServer
  AOI (EPSG:2285): 1254298,289554  →  1276244,305537
  RGB already exists, skipping: /content/drive/MyDrive/treedata/Full_Image/SnoCo/v2/snoco_2016_rgb.tif
  NIR already exists, skipping: /content/drive/MyDrive/treedata/Full_Image/SnoCo/v2/snoco_2016_nir.tif

  All downloads complete.

